# Projeto Final — Aprendizado de Máquina Supervisionado
## Notebook de Apoio: Validação Automatizada do "Gate de Sanidade"

**Disciplina:** *Aprendizado de Máquina Supervisionado (MC886 / IC-UNICAMP)*  
**Professor:** *Marcos Medeiros Raimundo*  
**Semestre:** *2º Semestre de 2026*  

[← Voltar ao Enunciado do Projeto Final](./projeto.qmd)


| Nome | RA |
|---|---|
| *Integrante 1* | *RA 1* |
| *Integrante 2* | *RA 2* |
| *Integrante 3 (se houver)* | *RA 3* |

> **Composição da equipe:** 2 ou 3 alunos, impreterivelmente (trabalhos individuais ou grupos com 4+ não são permitidos).


---

### Objetivo deste Notebook

Conforme estipulado no [Enunciado Oficial do Projeto Final](./projeto.qmd), antes de iniciar a modelagem, a equipe deve validar se o conjunto de dados escolhido cumpre os **Critérios de Não-Trivialidade e Gate de Sanidade**:

1. **Volume Amostral e Dimensionalidade:** $N \ge 1.000$ instâncias e $d \ge 8$ atributos preditivos informativos.
2. **Proibição de Bases de Brinquedo:** Não são aceitas bases clássicas/artificiais (*Iris*, *Titanic* básico, *Boston Housing*, *Wine Quality*, *Breast Cancer Wisconsin*).
3. **Margem para Aprendizado (Não-Trivialidade):** O estimador linear básico inicial (*baseline* regularizado) **não pode atingir acurácia, F1-Score ou ROC-AUC superior a $0{,}92$**. Se um modelo linear ingênuo já atinge performance quase perfeita, o problema é estatisticamente trivial e não permite investigar os mecanismos de viés-variância e ensembles.

> **Importante:** Este notebook fornece a função de validação automatizada `check_dataset_gate(...)`. Abaixo, demonstramos a aplicação dessa função em uma base real que **NÃO PASSA NO TESTE** (para exemplificar o diagnóstico de reprovação explícito), e deixamos a estrutura pronta para você testar a base da sua equipe.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [2]:
TOY_DATASETS_BLACKLIST = [
    "iris", "titanic", "boston", "boston_housing", 
    "wine", "wine_quality", "breast_cancer", "cancer", "diabetes_simple"
]

def check_dataset_gate(
    X: pd.DataFrame | np.ndarray,
    y: pd.Series | np.ndarray,
    dataset_name: str = "Meu_Dataset",
    task_type: str = "classification",
    max_metric_threshold: float = 0.92,
    min_samples: int = 1000,
    min_features: int = 8,
) -> dict:
    """
    Verifica se o conjunto de dados atende aos requisitos do Gate de Sanidade do Projeto Final.
    """
    N = len(X)
    d = X.shape[1] if hasattr(X, "shape") else len(X[0])
    
    failures = []
    
    # 1. Checagem de Volume e Dimensionalidade
    if N < min_samples:
        failures.append(f"Volume amostral insuficiente: N = {N:,} (mínimo exigido: {min_samples:,}).")
    if d < min_features:
        failures.append(f"Dimensionalidade insuficiente: d = {d} atributos (mínimo exigido: {min_features}).")
        
    # 2. Checagem de Lista Negra (Toy Datasets)
    normalized_name = dataset_name.lower().replace(" ", "_").replace("-", "_")
    for blacklisted in TOY_DATASETS_BLACKLIST:
        if blacklisted in normalized_name:
            failures.append(f"Base de brinquedo/clássica proibida identificada: '{dataset_name}'.")
            break
            
    # 3. Checagem de Não-Trivialidade (Baseline Linear)
    if task_type == "classification":
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=42))
        ])
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_res = cross_validate(pipe, X, y, cv=cv, scoring=["accuracy", "f1_weighted", "roc_auc"])
        
        acc = float(np.mean(cv_res["test_accuracy"]))
        f1 = float(np.mean(cv_res["test_f1_weighted"]))
        auc = float(np.mean(cv_res["test_roc_auc"])) if not np.isnan(cv_res["test_roc_auc"]).any() else acc
        
        max_metric = max(acc, f1, auc)
        metric_name = "ROC-AUC" if max_metric == auc else ("F1" if max_metric == f1 else "Acurácia")
        
        if max_metric > max_metric_threshold:
            failures.append(
                f"Problema estatisticamente trivial: Baseline linear atingiu {metric_name} = {max_metric:.3f} "
                f"(limite máximo permitido: {max_metric_threshold:.2f})."
            )
    else:
        from sklearn.linear_model import Ridge
        pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
        cv_res = cross_validate(pipe, X, y, cv=5, scoring=["r2"])
        r2 = float(np.mean(cv_res["test_r2"]))
        max_metric = r2
        metric_name = "R²"
        if r2 > max_metric_threshold:
            failures.append(
                f"Problema trivial: Baseline linear atingiu R² = {r2:.3f} "
                f"(limite máximo permitido: {max_metric_threshold:.2f})."
            )
            
    # Exibição do Laudo Formatado
    print("=" * 72)
    print(f"RELATÓRIO DE AUDITORIA — GATE DE SANIDADE: {dataset_name}")
    print("=" * 72)
    print(f"• Dimensões observadas : N = {N:,} amostras, d = {d} atributos")
    if task_type == "classification":
        print(f"• Desempenho Baseline  : Acurácia = {acc:.3f} | F1 = {f1:.3f} | ROC-AUC = {auc:.3f}")
    else:
        print(f"• Desempenho Baseline  : R² = {r2:.3f}")
    print("-" * 72)
    
    passed = len(failures) == 0
    if passed:
        print("✅ APROVADO NO GATE DE SANIDADE!")
        print("O conjunto de dados é perfeitamente elegível para o Projeto Final.")
        print("Ele apresenta dimensões adequadas e desafio estatístico suficiente.")
    else:
        print("❌ REPROVADO NO GATE DE SANIDADE! (ESTA BASE NÃO PASSA NO TESTE)")
        print("\nMotivos formais da reprovação:")
        for idx, fail in enumerate(failures, 1):
            print(f"  {idx}. [FALHA] {fail}")
        print("\n⚠️ AÇÃO NECESSÁRIA:")
        print("Esta base NÃO pode ser utilizada no Projeto Final da disciplina.")
        print("A equipe deve selecionar um conjunto de dados do mundo real com N >= 1.000")
        print("e complexidade suficiente para que a baseline linear não atinja teto quase perfeito.")
    print("=" * 72)
    
    return {
        "passed": passed,
        "N": N,
        "d": d,
        "failures": failures,
        "max_metric": max_metric
    }

---
## 1. Demonstração: Testando uma base que NÃO PASSA no teste

Abaixo, submetemos a base clássica **Breast Cancer Wisconsin** (disponível nativamente no scikit-learn).  
Ela ilustra com clareza a reprovação por violar múltiplos requisitos do projeto ao mesmo tempo:
- Volume amostral menor que o mínimo ($N = 569 < 1.000$);
- Base clássica de brinquedo listada na restrição;
- Problema linearmente separável / quase trivial (a regressão logística obtém ROC-AUC de $0{,}994$, estourando o teto de $0{,}92$).

In [3]:
# Carregando a base de demonstração (que falha no teste)
cancer_data = load_breast_cancer(as_frame=True)
X_cancer = cancer_data.data
y_cancer = cancer_data.target

# Executando a auditoria oficial do Gate de Sanidade
resultado_cancer = check_dataset_gate(
    X=X_cancer,
    y=y_cancer,
    dataset_name="Breast Cancer Wisconsin",
    task_type="classification"
)

RELATÓRIO DE AUDITORIA — GATE DE SANIDADE: Breast Cancer Wisconsin
• Dimensões observadas : N = 569 amostras, d = 30 atributos
• Desempenho Baseline  : Acurácia = 0.975 | F1 = 0.975 | ROC-AUC = 0.994
------------------------------------------------------------------------
❌ REPROVADO NO GATE DE SANIDADE! (ESTA BASE NÃO PASSA NO TESTE)

Motivos formais da reprovação:
  1. [FALHA] Volume amostral insuficiente: N = 569 (mínimo exigido: 1,000).
  2. [FALHA] Base de brinquedo/clássica proibida identificada: 'Breast Cancer Wisconsin'.
  3. [FALHA] Problema estatisticamente trivial: Baseline linear atingiu ROC-AUC = 0.994 (limite máximo permitido: 0.92).

⚠️ AÇÃO NECESSÁRIA:
Esta base NÃO pode ser utilizada no Projeto Final da disciplina.
A equipe deve selecionar um conjunto de dados do mundo real com N >= 1.000
e complexidade suficiente para que a baseline linear não atinja teto quase perfeito.


---
## 2. Como validar a base de dados da sua equipe

Para validar a base proposta pela sua equipe (obtida no Kaggle, UCI, OpenML, Hugging Face ou órgãos de dados abertos), utilize o modelo abaixo.  
Basta carregar os dados via Pandas e chamar a função `check_dataset_gate`.

Inclua o relatório de auditoria gerado na **Seção 1 do seu Relatório Técnico**.


In [ ]:
# Roteiro para a sua equipe executar:
# 1. Carregue seu arquivo de dados:
# df = pd.read_csv("data/meu_dataset.csv")

# 2. Isole as matrizes X e y:
# X_equipe = df.drop(columns=["coluna_alvo"])
# y_equipe = df["coluna_alvo"]

# 3. Execute a auditoria:
# resultado = check_dataset_gate(
#     X=X_equipe,
#     y=y_equipe,
#     dataset_name="Nome_Do_Seu_Dataset",
#     task_type="classification"  # ou "regression"
# )
